# M2 Showcase — Perfect-Foresight Dispatch, HB_HOUSTON June 2025

Proof of the perfect-foresight LP benchmark from M2 running against real ingested data. Not a polished report (that's M5) — this shows the actual dispatch trajectory the optimizer finds, and how the three variants (RTM-only, DAM-only, DAM+AS co-optimized) compare.

See [`docs/milestones/M2-perfect-foresight.md`](../docs/milestones/M2-perfect-foresight.md) for the full write-up, and [ADR 0006](../docs/adr/0006-perfect-foresight-lp-formulation.md) / [ADR 0007](../docs/adr/0007-as-capacity-only-co-optimization.md) for the formulation and assumptions.

**Contents:**
1. Revenue by variant (RTM-only, DAM-only, DAM+AS)
2. Dispatch trajectory for one week — charge/discharge/SoC against price
3. A single day zoomed in, to see the buy-low/sell-high pattern clearly
4. AS capacity awards by product (DAM+AS variant)
5. Comparison against M1's naive teaser number

In [1]:
import os
import sys
from pathlib import Path

import duckdb
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.templates.default = "plotly_white"

if not (Path.cwd() / "data" / "ercot.duckdb").exists():
    os.chdir(Path.cwd().parent)
sys.path.insert(0, "src")

from ercot_bess.models.battery import BatterySpec
from ercot_bess.optimize.data_loading import load_dam_as_price_series, load_spp_series
from ercot_bess.optimize.perfect_foresight import solve_perfect_foresight

con = duckdb.connect("data/ercot.duckdb", read_only=True)
con.execute("PRAGMA disable_progress_bar")

battery = BatterySpec()
battery

BatterySpec(power_mw=100.0, energy_mwh=200.0, round_trip_efficiency=0.86, min_soc_fraction=0.0, max_soc_fraction=1.0, daily_cycle_limit=1.5, degradation_cost_per_mwh=2.0)

## 1. Revenue by variant

Solving all three perfect-foresight variants over the full month, HB_HOUSTON, default 100MW/200MWh battery.

In [2]:
import datetime as dt

HUB = "HB_HOUSTON"
START, END = dt.date(2025, 6, 1), dt.date(2025, 6, 30)

rtm_ts, rtm_prices = load_spp_series(con, "silver_rtm_spp", HUB, START, END)
rtm_result = solve_perfect_foresight(rtm_ts, rtm_prices, battery, interval_hours=0.25)

dam_ts, dam_prices = load_spp_series(con, "silver_dam_spp", HUB, START, END)
dam_result = solve_perfect_foresight(dam_ts, dam_prices, battery, interval_hours=1.0)

as_prices = load_dam_as_price_series(con, dam_ts, START, END)
dam_as_result = solve_perfect_foresight(
    dam_ts, dam_prices, battery, interval_hours=1.0, as_prices_usd_per_mw=as_prices
)

summary = pd.DataFrame([
    {"variant": "RTM-only", "revenue_usd": rtm_result.total_revenue_usd},
    {"variant": "DAM-only", "revenue_usd": dam_result.total_revenue_usd},
    {"variant": "DAM + AS", "revenue_usd": dam_as_result.total_revenue_usd},
])

fig = go.Figure(go.Bar(x=summary["variant"], y=summary["revenue_usd"],
                        marker_color=["#636EFA", "#EF553B", "#00CC96"],
                        text=summary["revenue_usd"].map(lambda v: f"${v:,.0f}"), textposition="outside"))
fig.update_layout(title="Perfect-foresight revenue by variant — June 2025, HB_HOUSTON",
                   yaxis_title="$", height=400)
fig.show()
summary

/mnt/c/Users/ukule/PycharmProjects/ercot-bess-lab/.claude/worktrees/m2-perfect-foresight/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:83: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


,variant,revenue_usd
0,RTM-only,371952.717743
1,DAM-only,244165.638212
2,DAM + AS,517092.270809


## 2. Dispatch trajectory — one week (RTM-only)

Price, dispatch power (charge negative / discharge positive), and state of charge, stacked so the "buy low, sell high, respect SoC limits" pattern is visible directly.

In [3]:
week_start, week_end = dt.datetime(2025, 6, 15), dt.datetime(2025, 6, 22)
idx = [i for i, ts in enumerate(rtm_ts) if week_start <= ts.replace(tzinfo=None) < week_end]

ts_week = [rtm_ts[i] for i in idx]
price_week = [rtm_prices[i] for i in idx]
net_power_week = [rtm_result.discharge_mw[i] - rtm_result.charge_mw[i] for i in idx]
soc_week = [rtm_result.soc_mwh[i] for i in idx]

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                     subplot_titles=("RTM price ($/MWh)", "Net dispatch power (MW, + = discharge)", "State of charge (MWh)"))
fig.add_trace(go.Scatter(x=ts_week, y=price_week, line=dict(color="#636EFA", width=1)), row=1, col=1)
fig.add_trace(go.Bar(x=ts_week, y=net_power_week, marker_color="#EF553B"), row=2, col=1)
fig.add_trace(go.Scatter(x=ts_week, y=soc_week, fill="tozeroy", line=dict(color="#00CC96", width=1)), row=3, col=1)
fig.add_hline(y=battery.max_soc_mwh, line_dash="dot", line_color="gray", row=3, col=1)
fig.update_layout(height=700, showlegend=False, title="RTM-only perfect-foresight dispatch — 2025-06-15 to 06-22")
fig.show()

## 3. Single day, zoomed in

Same three panels for one day, close enough to see individual 15-minute dispatch decisions.

In [4]:
day_start, day_end = dt.datetime(2025, 6, 18), dt.datetime(2025, 6, 19)
idx_day = [i for i, ts in enumerate(rtm_ts) if day_start <= ts.replace(tzinfo=None) < day_end]

ts_day = [rtm_ts[i] for i in idx_day]
price_day = [rtm_prices[i] for i in idx_day]
net_power_day = [rtm_result.discharge_mw[i] - rtm_result.charge_mw[i] for i in idx_day]
soc_day = [rtm_result.soc_mwh[i] for i in idx_day]

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                     subplot_titles=("RTM price ($/MWh)", "Net dispatch power (MW, + = discharge)", "State of charge (MWh)"))
fig.add_trace(go.Scatter(x=ts_day, y=price_day, line=dict(color="#636EFA", width=2), mode="lines+markers"), row=1, col=1)
fig.add_trace(go.Bar(x=ts_day, y=net_power_day, marker_color="#EF553B"), row=2, col=1)
fig.add_trace(go.Scatter(x=ts_day, y=soc_day, fill="tozeroy", line=dict(color="#00CC96", width=2)), row=3, col=1)
fig.add_hline(y=battery.max_soc_mwh, line_dash="dot", line_color="gray", row=3, col=1)
fig.update_layout(height=700, showlegend=False, title="RTM-only perfect-foresight dispatch — 2025-06-18")
fig.show()

## 4. AS capacity awards (DAM + AS variant)

⚠️ Per [ADR 0007](../docs/adr/0007-as-capacity-only-co-optimization.md), this is capacity-only — no deployment energy, no SoC impact from being called. Expect awards close to full available headroom whenever the clearing price is positive, since there's no modeled downside to committing.

In [5]:
fig = go.Figure()
for product, award in dam_as_result.as_awards_mw.items():
    fig.add_trace(go.Scatter(x=dam_ts, y=award, name=product, stackgroup="as", mode="none"))
fig.add_hline(y=battery.power_mw, line_dash="dot", line_color="gray",
              annotation_text="battery power rating")
fig.update_layout(title="DAM AS capacity awards by product — June 2025",
                   yaxis_title="MW awarded", height=450, legend=dict(orientation="h", y=1.08))
fig.show()

as_award_totals = {p: round(sum(v), 1) for p, v in dam_as_result.as_awards_mw.items()}
as_award_totals

{'ECRS': np.float64(5003.4),
 'NonSpin': np.float64(6872.6),
 'RRS': np.float64(500.0),
 'RegDown': np.float64(63113.1),
 'RegUp': np.float64(53959.3)}

## 5. Comparison to M1's naive teaser

M1's showcase notebook computed a deliberately naive "sort each day's RTM prices, charge at the 8 cheapest 15-min intervals, discharge at the 8 priciest" number: **$391,354** for the month. The real LP's RTM-only result is *lower* — not a bug. The naive calculation assumed frictionless 100% round-trip efficiency and zero degradation cost; the real LP applies the actual `BatterySpec` defaults (86% RTE, $2/MWh degradation). A physically realistic ceiling is necessarily below a frictionless one.

In [6]:
naive_m1_estimate = 391354
comparison = pd.DataFrame([
    {"method": "M1 naive teaser (frictionless, RTM)", "revenue_usd": naive_m1_estimate},
    {"method": "M2 real LP (86% RTE, $2/MWh degradation, RTM)", "revenue_usd": rtm_result.total_revenue_usd},
])
fig = go.Figure(go.Bar(x=comparison["method"], y=comparison["revenue_usd"],
                        marker_color=["#AB63FA", "#636EFA"],
                        text=comparison["revenue_usd"].map(lambda v: f"${v:,.0f}"), textposition="outside"))
fig.update_layout(title="Naive frictionless teaser vs. real perfect-foresight LP", yaxis_title="$", height=400)
fig.show()

pct_of_naive = rtm_result.total_revenue_usd / naive_m1_estimate * 100
print(f"Real LP captures {pct_of_naive:.1f}% of the naive frictionless estimate — "
      f"the ~{100 - pct_of_naive:.1f}% gap is exactly what efficiency losses and degradation cost account for.")

Real LP captures 95.0% of the naive frictionless estimate — the ~5.0% gap is exactly what efficiency losses and degradation cost account for.


## Next: M3

This LP has full hindsight — it's the ceiling, not a strategy. M3 builds the causal (no-lookahead) strategies — DA-committed and rolling-horizon MPC — behind a walk-forward backtest engine, and finally puts a numerator over this denominator: **% of perfect revenue captured**.